In [1]:
from itscalledsoccer.client import AmericanSoccerAnalysis
from plotnine import *
from colour import Color
import numpy as np
import pandas as pd

asa_client = AmericanSoccerAnalysis()

In [7]:
def decorate_team_ids(df):
    team_ids = set()

    for c in df.columns:
        if "team_id" in c:
            team_ids |= set(df[c].unique())

    teams_df = asa_client.get_teams(ids=list(team_ids))
    team_map = dict(zip(teams_df.team_id, teams_df.team_abbreviation))

    for c in df.columns:
        if "team_id" in c:
            df[c.replace("team_id", "team")] = df[c].map(team_map)
            df[c.replace("team_id", "team_logo")] = (
                "https://american-soccer-analysis-headshots.s3.amazonaws.com/club_logos/"
                + df[c]
                + ".png"
            )

    return df


def decorate_player_ids(df):
    player_ids = set()

    for c in df.columns:
        if "player_id" in c:
            player_ids |= set(df[c].unique())

    players_df = asa_client.get_players(ids=list(player_ids))
    player_map = dict(zip(players_df.player_id, players_df.player_name))

    for c in df.columns:
        if "player_id" in c:
            df[c.replace("player_id", "player")] = df[c].map(player_map)
            df[c.replace("player_id", "player_headshot")] = (
                "https://american-soccer-analysis-headshots.s3.amazonaws.com/player_headshots/"
                + df[c]
                + ".png"
            )

    return df

In [8]:
asa_client.get_teams(leagues="nwsl")

,team_id,team_name,team_short_name,team_abbreviation,competition
0,315VnJ759x,Bay FC,Bay FC,BAY,nwsl
1,4JMAk47qKg,Houston Dash,Houston,HOU,nwsl
2,4wM4Ezg5jB,Boston Breakers,Boston,BOS,nwsl
3,4wM4rZdqjB,Kansas City Current,Kansas City,KC,nwsl
4,7vQ7BBzqD1,Seattle Reign FC,Seattle,SEA,nwsl
5,7VqG1lYMvW,San Diego Wave FC,San Diego,SD,nwsl
6,aDQ0lzvQEv,Washington Spirit,Washington,WAS,nwsl
7,eV5D2w9QKn,Utah Royals FC,Utah Royals,UTA,nwsl
8,eV5DR6YQKn,Racing Louisville FC,Louisville,LOU,nwsl
9,KPqjw8PQ6v,Chicago Stars FC,Chicago,CHI,nwsl


In [9]:
games_df = asa_client.get_games(
    seasons="2025", leagues="nwsl", team_names="Racing Louisville FC"
)

In [10]:
games_df

,game_id,date_time_utc,home_score,away_score,home_team_id,away_team_id,referee_id,stadium_id,home_manager_id,away_manager_id,expanded_minutes,season_name,matchday,attendance,knockout_game,status,last_updated_utc
0,7vQ7bja2qD,2025-05-10 00:00:00 UTC,1,0,eV5DR6YQKn,raMyrr25d2,KAqBb1vqbg,BLMvra8Mxe,kRQa4GoqKZ,9vQ2r1LQK6,104,2025,8,5495,False,FullTime,2025-05-10 02:40:35 UTC
1,9z5kjk9jMA,2025-05-03 01:58:00 UTC,1,2,4JMAk47qKg,eV5DR6YQKn,gOMno2X5wN,0x5g6ojM7O,KPqjdAEM6v,kRQa4GoqKZ,99,2025,7,5765,False,FullTime,2025-05-10 00:22:22 UTC
2,2lqRpdZ0Mr,2025-04-27 20:00:00 UTC,3,3,Pk5LeeNqOW,eV5DR6YQKn,jYQJ3wEMGR,p6qbX06M0G,XVqKEk0M01,kRQa4GoqKZ,110,2025,6,16219,False,FullTime,2025-05-04 21:20:53 UTC
3,jYQJmLDPqG,2025-04-19 19:00:00 UTC,1,4,eV5DR6YQKn,7VqG1lYMvW,KPqj32nQ6v,BLMvra8Mxe,kRQa4GoqKZ,zeQZzPgQKw,105,2025,5,4431,False,FullTime,2025-04-24 14:48:57 UTC
4,NWMWp4Ej5l,2025-04-12 21:00:00 UTC,0,2,eV5DR6YQKn,aDQ0lzvQEv,2vQ1Bo8MrA,BLMvra8Mxe,kRQa4GoqKZ,9YqdzzjMvJ,102,2025,4,7886,False,FullTime,2025-04-18 15:23:13 UTC
5,jYQJmLDBqG,2025-03-30 22:00:00 UTC,0,1,KPqjw8PQ6v,eV5DR6YQKn,aDQ03rGqEv,9YqdkYR5vJ,2lqRGexQr0,kRQa4GoqKZ,101,2025,3,1806,False,FullTime,2025-04-05 02:37:52 UTC
6,4wM4bRykqj,2025-03-23 02:00:00 UTC,2,0,315VnJ759x,eV5DR6YQKn,ljqEyb4qx0,Vj58W84M8n,NPqxLwgq9d,kRQa4GoqKZ,107,2025,2,14742,False,FullTime,2025-03-25 13:51:36 UTC
7,9vQ2bnVYQK,2025-03-16 00:00:00 UTC,1,1,eV5DR6YQKn,zeQZeazqKw,e7Mz8Eeqr0,BLMvra8Mxe,kRQa4GoqKZ,p6qbJw3q0G,97,2025,1,2847,False,FullTime,2025-03-20 15:39:28 UTC


In [17]:
shots_df = decorate_player_ids(
    decorate_team_ids(
        asa_client._get_stats(
            game_id=list(games_df.game_id), type="shots", entity="games", leagues="nwsl"
        )
    )
)

shots_df = shots_df.merge(games_df[["game_id", "date_time_utc", "matchday"]])

In [18]:
shots_df

,game_id,period_id,expanded_minute,game_minute,team_id,shooter_player_id,assist_player_id,shot_location_x,shot_location_y,shot_end_location_x,...,pattern_of_play,shot_order,team,team_logo,shooter_player,shooter_player_headshot,assist_player,assist_player_headshot,date_time_utc,matchday
0,4wM4bRykqj,1,2,1,eV5DR6YQKn,2lqRk0vaQr,0Oq6wZexQ6,93.0,59.3,100.0,...,Set piece,31,LOU,https://american-soccer-analysis-headshots.s3....,Taylor Flint,https://american-soccer-analysis-headshots.s3....,Janine Sonis,https://american-soccer-analysis-headshots.s3....,2025-03-23 02:00:00 UTC,2
1,jYQJmLDBqG,1,3,2,KPqjw8PQ6v,aDQ0x3YEQE,zeQZXkwD5K,92.2,54.9,93.7,...,Corner,49,CHI,https://american-soccer-analysis-headshots.s3....,Hannah Anderson,https://american-soccer-analysis-headshots.s3....,Julia Grosso,https://american-soccer-analysis-headshots.s3....,2025-03-30 22:00:00 UTC,3
2,jYQJmLDBqG,1,3,2,KPqjw8PQ6v,vzqodLvBQa,zeQZXkwD5K,96.4,50.9,99.4,...,Corner,55,CHI,https://american-soccer-analysis-headshots.s3....,Allyson Schlegel,https://american-soccer-analysis-headshots.s3....,Julia Grosso,https://american-soccer-analysis-headshots.s3....,2025-03-30 22:00:00 UTC,3
3,jYQJmLDBqG,1,4,3,KPqjw8PQ6v,gpMOoGgDMz,7vQ7BOOLqD,92.6,60.9,99.5,...,Regular,63,CHI,https://american-soccer-analysis-headshots.s3....,Samantha Staab,https://american-soccer-analysis-headshots.s3....,Justina Gaynor,https://american-soccer-analysis-headshots.s3....,2025-03-30 22:00:00 UTC,3
4,2lqRpdZ0Mr,1,4,3,eV5DR6YQKn,odMXGw2oQY,0,86.3,45.2,100.0,...,Regular,71,LOU,https://american-soccer-analysis-headshots.s3....,Kayla Fischer,https://american-soccer-analysis-headshots.s3....,NaN,https://american-soccer-analysis-headshots.s3....,2025-04-27 20:00:00 UTC,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,jYQJmLDPqG,2,101,91,7VqG1lYMvW,0x5gPXRRq7,XVqK28bB50,86.1,37.9,100.0,...,Regular,1772,SD,https://american-soccer-analysis-headshots.s3....,Makenzy Robbe,https://american-soccer-analysis-headshots.s3....,Chiamaka Okwuchukwu,https://american-soccer-analysis-headshots.s3....,2025-04-19 19:00:00 UTC,5
194,9vQ2bnVYQK,2,96,92,zeQZeazqKw,NWMWl2jY5l,4wM4aOLA5j,70.9,47.7,100.0,...,Regular,1797,NC,https://american-soccer-analysis-headshots.s3....,Kaleigh Kurtz,https://american-soccer-analysis-headshots.s3....,Malia Berkely,https://american-soccer-analysis-headshots.s3....,2025-03-16 00:00:00 UTC,1
195,jYQJmLDPqG,2,104,94,eV5DR6YQKn,KPqjKkoPQ6,0,85.2,55.0,88.1,...,Set piece,1834,LOU,https://american-soccer-analysis-headshots.s3....,Savannah DeMelo,https://american-soccer-analysis-headshots.s3....,NaN,https://american-soccer-analysis-headshots.s3....,2025-04-19 19:00:00 UTC,5
196,jYQJmLDPqG,2,104,94,eV5DR6YQKn,0Oq6wZexQ6,0,72.6,60.9,84.5,...,Set piece,1837,LOU,https://american-soccer-analysis-headshots.s3....,Janine Sonis,https://american-soccer-analysis-headshots.s3....,NaN,https://american-soccer-analysis-headshots.s3....,2025-04-19 19:00:00 UTC,5


In [19]:
shots_df.T

,0,1,2,3,4,5,6,7,8,9,...,188,189,190,191,192,193,194,195,196,197
game_id,4wM4bRykqj,jYQJmLDBqG,jYQJmLDBqG,jYQJmLDBqG,2lqRpdZ0Mr,9vQ2bnVYQK,7vQ7bja2qD,9vQ2bnVYQK,NWMWp4Ej5l,7vQ7bja2qD,...,jYQJmLDBqG,9z5kjk9jMA,jYQJmLDPqG,9z5kjk9jMA,jYQJmLDPqG,jYQJmLDPqG,9vQ2bnVYQK,jYQJmLDPqG,jYQJmLDPqG,jYQJmLDPqG
period_id,1,1,1,1,1,1,1,1,1,1,...,2,2,2,2,2,2,2,2,2,2
expanded_minute,2,3,3,4,4,3,5,5,8,8,...,98,97,96,98,98,101,96,104,104,105
game_minute,1,2,2,3,3,2,4,4,7,7,...,93,94,86,95,88,91,92,94,94,95
team_id,eV5DR6YQKn,KPqjw8PQ6v,KPqjw8PQ6v,KPqjw8PQ6v,eV5DR6YQKn,eV5DR6YQKn,eV5DR6YQKn,zeQZeazqKw,aDQ0lzvQEv,eV5DR6YQKn,...,KPqjw8PQ6v,4JMAk47qKg,eV5DR6YQKn,4JMAk47qKg,eV5DR6YQKn,7VqG1lYMvW,zeQZeazqKw,eV5DR6YQKn,eV5DR6YQKn,eV5DR6YQKn
shooter_player_id,2lqRk0vaQr,aDQ0x3YEQE,vzqodLvBQa,gpMOoGgDMz,odMXGw2oQY,KPqjKkoPQ6,2lqRkmXAQr,gpMOoXeOMz,NWMWgzp7Ml,KPqjKkoPQ6,...,EGMPZBdjqa,e7MzJRO7Mr,2lqRkmXAQr,odMX7rK2QY,odMXRl80qY,0x5gPXRRq7,NWMWl2jY5l,KPqjKkoPQ6,0Oq6wZexQ6,0Oq6wZexQ6
assist_player_id,0Oq6wZexQ6,zeQZXkwD5K,zeQZXkwD5K,7vQ7BOOLqD,0,NWMWV0NYQl,0,9Yqdkgdm5v,0,2lqRkmXAQr,...,KPqjBOaNq6,p6qboZk4Q0,KPqjKkoPQ6,0Oq6wGnrQ6,odMXGw2oQY,XVqK28bB50,4wM4aOLA5j,0,0,KPqjKkoPQ6
shot_location_x,93.0,92.2,96.4,92.6,86.3,80.6,68.3,74.9,75.4,69.8,...,75.9,75.0,78.7,83.8,79.1,86.1,70.9,85.2,72.6,80.2
shot_location_y,59.3,54.9,50.9,60.9,45.2,66.9,36.7,42.5,54.2,41.6,...,45.2,58.9,44.3,69.6,31.2,37.9,47.7,55.0,60.9,63.9
shot_end_location_x,100.0,93.7,99.4,99.5,100.0,100.0,100.0,97.5,98.9,98.5,...,100.0,100.0,89.1,90.2,87.2,100.0,100.0,88.1,84.5,100.0


In [ ]:
shots_df.groupby(["game_id", "team", "matchday"])[
    ["shot_xg", "shot_psxg", "goal"]
].sum().sort_values("matchday")

shot_xg  shot_psxg  goal
game_id    team matchday                          
9vQ2bnVYQK LOU  1          1.5053     1.9562     1
           NC   1          1.2140     0.8070     1
4wM4bRykqj BAY  2          0.4782     1.0644     2
           LOU  2          0.9722     0.4221     0
jYQJmLDBqG CHI  3          0.8540     0.3360     0
           LOU  3          2.3394     1.9335     1
NWMWp4Ej5l LOU  4          0.6772     0.5342     0
           WAS  4          0.7407     1.4316     2
jYQJmLDPqG LOU  5          1.4392     1.0022     1
           SD   5          0.9270     0.8655     4
2lqRpdZ0Mr LOU  6          3.0995     4.2988     3
           POR  6          2.3802     1.7284     3
9z5kjk9jMA HOU  7          1.1030     1.5975     1
           LOU  7          0.6770     1.1054     2
7vQ7bja2qD LOU  8          0.4890     0.3573     1
           NJY  8          0.8202     0.2352     0